# Redact PDFs, including scans, using the ASE analyzer

Render every page, OCR the whole page (including embedded images), analyze OCR and native PDF text through your ASE analyzer, and paint detected regions black. Write a **fresh image-only PDF**. Original text layers, attachments, forms, links, annotations, scripts, document metadata, and hidden objects are not copied. Visible annotations/forms are rendered into the page before detection.

This intentionally trades searchability, editability, accessibility tags, and vector quality for clean rebuilt pages. Detection/OCR can miss PII; output is **not certified PII-free**. No local Presidio engines or anonymizer service are needed.

Copy this notebook **and `ase_document_tools.py`** to the kernel host. Install dependencies in the selected environment:
```powershell
uv pip install --python .venv-ase\Scripts\python.exe requests azure-identity ipykernel Pillow "PyMuPDF>=1.24,<2"
```
**PyMuPDF uses AGPL/commercial licensing; confirm the appropriate license for your use before installation.** Install **Tesseract OCR and language data** on the VM separately; a Python package does not install the engine. Use Python 3.10 through 3.14 and an updated, trusted processing host.

Start with synthetic files. Never display original text, tokens, raw HTTP error bodies, or pages containing real PII in notebook output. Clear outputs before sharing and protect both documents.


In [ ]:
import os
import sys
from pathlib import Path

TOOLS_DIRECTORY = Path(os.environ.get("ASE_SAMPLE_DIRECTORY", str(Path.cwd())))
ANALYZER_BASE_URL = os.environ.get("ANALYZER_BASE_URL", "https://presidio-analyzer.ase-jtl.appserviceenvironment.us")
ANALYZER_API_RESOURCE = os.environ.get("ANALYZER_API_RESOURCE", "")
MANAGED_IDENTITY_CLIENT_ID = os.environ.get("MANAGED_IDENTITY_CLIENT_ID", "")
INPUT_DOCUMENT = Path(os.environ.get("INPUT_DOCUMENT", str(Path("sample_data") / "sample.pdf")))
OUTPUT_DOCUMENT = Path(os.environ.get("OUTPUT_DOCUMENT", str(INPUT_DOCUMENT.with_name(INPUT_DOCUMENT.stem + "_redacted.pdf"))))
LANGUAGE = "en"
OCR_LANGUAGE = "eng"
OCR_MIN_CONFIDENCE = 50
TESSERACT_COMMAND = os.environ.get("TESSERACT_COMMAND", "tesseract")
RENDER_DPI = 200


Set the analyzer's registered Application ID URI without `/.default`, your calling UMI client ID, helper folder, and input/output paths. Keep the working Government v1 issuer `https://login.microsoftonline.us/<tenant-id>/` without `/v2.0`, tenant/caller restrictions, and TLS verification. Use `REQUESTS_CA_BUNDLE` if your private CA requires it.

`OCR_LANGUAGE` is the installed Tesseract language (for example `eng`); `LANGUAGE` is the analyzer ISO code (`en`). Set `TESSERACT_COMMAND` to an absolute executable path if needed. Use new output filenames in existing directories. Rendering requires at least 144 DPI; the sample limits input to 100 MiB, 200 pages, and 25 million pixels per page.


In [ ]:
if not (TOOLS_DIRECTORY / "ase_document_tools.py").is_file():
    raise FileNotFoundError("Copy ase_document_tools.py to the kernel host and set TOOLS_DIRECTORY to its folder.")
sys.path.insert(0, str(TOOLS_DIRECTORY.resolve()))
from ase_document_tools import ASEAnalyzer, redact_pdf

with ASEAnalyzer(ANALYZER_BASE_URL, ANALYZER_API_RESOURCE, MANAGED_IDENTITY_CLIENT_ID, LANGUAGE) as analyzer:
    summary = redact_pdf(
        INPUT_DOCUMENT, OUTPUT_DOCUMENT, analyzer.analyze, dpi=RENDER_DPI,
        ocr_language=OCR_LANGUAGE, tesseract=TESSERACT_COMMAND,
        min_confidence=OCR_MIN_CONFIDENCE,
    )
print(f"Saved image-only PDF: {summary['pages']} pages; {summary['text_detections']} native-text and {summary['ocr_detections']} OCR detections. Review for missed PII.")


## Review and failure handling

Whole detected OCR words are covered with a small pixel margin, so partial-word matches may over-redact. Native character geometry is also analyzed, with rotation mapping, to supplement OCR on text PDFs. OCR and native counts may overlap; they are not unique PII counts.

Missing Tesseract/language data, low OCR confidence, invalid geometry, no OCR text on a nonuniform page, encrypted PDFs, or service failures refuse output. Even confident OCR may be wrong or miss words. Handwriting, unusual orientations/fonts, photographs, and complex scans need careful private review. Uniform blank pages may pass without text.

The helper does not save rendered originals or recognized text to disk. Publication uses a same-directory hard link so an existing output cannot be replaced. Your filesystem must support hard links. Input is never modified. Verify synthetic outputs by checking both rendered pixels and the absence of original searchable text/attachments/metadata; a success count alone is insufficient.

Authentication failures: check the registered resource, attached identity client ID, audience/issuer version, and App Service caller/network restrictions. Keep TLS enabled and do not broaden permissions to hide an error.
